<a href="https://colab.research.google.com/github/Simge-bit/-dev1/blob/main/%C4%B0lk_Model_e%C4%9Fitimi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers datasets accelerate

In [ ]:
import os
print(os.listdir("/content"))

['.config', 'items.csv', 'training_pairs.csv', 'terms.csv', 'drive', 'sample_data']


In [4]:
import pandas as pd
import numpy as np

train = pd.read_csv("/content/training_pairs.csv")
items = pd.read_csv("/content/items.csv")
terms = pd.read_csv("/content/terms.csv")

positive = train.copy()
positive['label'] = 1

negative = train.copy()
negative['item_id'] = train['item_id'].sample(frac=1, random_state=42).values
negative['label'] = 0

df = pd.concat([positive, negative]).reset_index(drop=True)
df = df.merge(terms, on='term_id', how='left')
df = df.merge(items[['item_id', 'title', 'category']], on='item_id', how='left')

df_small = df.sample(n=10000, random_state=42).reset_index(drop=True)
print("Veri hazır:", len(df_small))
print(df_small['label'].value_counts())

ParserError: Error tokenizing data. C error: EOF inside string starting at row 218314

In [1]:
!pip install kaggle

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"simgesolmaz","key":"4033e582c29fe65e8bbde4664f84baf3"}'}

In [3]:
import os
os.makedirs("/root/.kaggle", exist_ok=True)
os.rename("/content/kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)
print("Kaggle hazır!")

Kaggle hazır!


In [4]:
!kaggle competitions download -c trendyol-e-ticaret-yarismasi-2026-kaggle -p /content/

100% 211M/211M [00:05<00:00, 43.4MB/s]



In [5]:
!unzip /content/trendyol-e-ticaret-yarismasi-2026-kaggle.zip -d /content/

Archive:  /content/trendyol-e-ticaret-yarismasi-2026-kaggle.zip
  inflating: /content/items.csv      
  inflating: /content/sample_submission.csv  
  inflating: /content/submission_pairs.csv  
  inflating: /content/terms.csv      
  inflating: /content/training_pairs.csv  


In [6]:
import pandas as pd
import numpy as np

train = pd.read_csv("/content/training_pairs.csv")
items = pd.read_csv("/content/items.csv")
terms = pd.read_csv("/content/terms.csv")

positive = train.copy()
positive['label'] = 1

negative = train.copy()
negative['item_id'] = train['item_id'].sample(frac=1, random_state=42).values
negative['label'] = 0

df = pd.concat([positive, negative]).reset_index(drop=True)
df = df.merge(terms, on='term_id', how='left')
df = df.merge(items[['item_id', 'title', 'category']], on='item_id', how='left')

df_small = df.sample(n=10000, random_state=42).reset_index(drop=True)
print("Veri hazır:", len(df_small))
print(df_small['label'].value_counts())

Veri hazır: 10000
label
0    5026
1    4974
Name: count, dtype: int64


In [7]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

examples = []
for _, row in df_small.iterrows():
    text = row['query'] + " [SEP] " + str(row['title']) + " " + str(row['category'])
    examples.append(InputExample(texts=[text, text], label=float(row['label'])))

model_st = SentenceTransformer("emrecan/bert-base-turkish-cased-mean-nli-stsb-tr")
dataloader = DataLoader(examples, shuffle=True, batch_size=16)
loss = losses.CosineSimilarityLoss(model_st)

model_st.fit(
    train_objectives=[(dataloader, loss)],
    epochs=1,
    warmup_steps=100,
    show_progress_bar=True
)

model_st.save("/content/finetuned-model")
print("Eğitim bitti, model kaydedildi!")

/tmp/ipykernel_549/1916561368.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.157206


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Eğitim bitti, model kaydedildi!


In [8]:
from sentence_transformers import SentenceTransformer, util

model_test = SentenceTransformer("/content/finetuned-model")

cumle1 = "spor ayakkabı"
cumle2 = "nike koşu ayakkabısı"
cumle3 = "bebek bezi"

e1 = model_test.encode(cumle1)
e2 = model_test.encode(cumle2)
e3 = model_test.encode(cumle3)

print("spor ayakkabı - nike koşu ayakkabısı:", round(util.cos_sim(e1, e2).item(), 2))
print("spor ayakkabı - bebek bezi:", round(util.cos_sim(e1, e3).item(), 2))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

spor ayakkabı - nike koşu ayakkabısı: 0.15
spor ayakkabı - bebek bezi: 0.23


In [9]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

examples = []
for _, row in df_small.iterrows():
    query_text = str(row['query'])
    item_text = str(row['title']) + " " + str(row['category'])
    examples.append(InputExample(texts=[query_text, item_text], label=float(row['label'])))

model_st = SentenceTransformer("emrecan/bert-base-turkish-cased-mean-nli-stsb-tr")
dataloader = DataLoader(examples, shuffle=True, batch_size=16)
loss = losses.CosineSimilarityLoss(model_st)

model_st.fit(
    train_objectives=[(dataloader, loss)],
    epochs=3,
    warmup_steps=100,
    show_progress_bar=True
)

model_st.save("/content/finetuned-model-v2")
print("Eğitim bitti!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.078850
1000,0.051146
1500,0.036477


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Eğitim bitti!


In [10]:
from sentence_transformers import SentenceTransformer, util

model_test = SentenceTransformer("/content/finetuned-model-v2")

e1 = model_test.encode("spor ayakkabı")
e2 = model_test.encode("nike koşu ayakkabısı erkek spor")
e3 = model_test.encode("bebek bezi yenidoğan")

print("spor ayakkabı - nike koşu ayakkabısı:", round(util.cos_sim(e1, e2).item(), 2))
print("spor ayakkabı - bebek bezi:", round(util.cos_sim(e1, e3).item(), 2))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

spor ayakkabı - nike koşu ayakkabısı: 0.95
spor ayakkabı - bebek bezi: -0.1


In [ ]:
# Bu sefer 100.000 satırla eğitelim
df_large = df.sample(n=100000, random_state=42).reset_index(drop=True)

examples2 = []
for _, row in df_large.iterrows():
    query_text = str(row['query'])
    item_text = str(row['title']) + " " + str(row['category'])
    examples2.append(InputExample(texts=[query_text, item_text], label=float(row['label'])))

model_st2 = SentenceTransformer("emrecan/bert-base-turkish-cased-mean-nli-stsb-tr")
dataloader2 = DataLoader(examples2, shuffle=True, batch_size=32)
loss2 = losses.CosineSimilarityLoss(model_st2)

model_st2.fit(
    train_objectives=[(dataloader2, loss2)],
    epochs=3,
    warmup_steps=200,
    show_progress_bar=True
)

model_st2.save("/content/finetuned-model-v3")
print("Eğitim bitti!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.081024
1000,0.057703
1500,0.049723
2000,0.046062
2500,0.042157
3000,0.041069
3500,0.035430
4000,0.031504
4500,0.031501
5000,0.030582


In [ ]:
sub = pd.read_csv("/content/submission_pairs.csv")
print(len(sub))